# Dataset Code Creation - Qualifying

In [ ]:
import fastf1 as ff1
import pandas as pd
from pathlib import Path

In [ ]:
def current_team_name(old_team_name):
    match old_team_name:
        case 'Toro Rosso' | 'AlphaTauri' | 'RB':
            return 'Racing Bulls'
        case 'Sauber' | 'Alfa Romeo Racing' | 'Alfa Romeo':
            return 'Kick Sauber'
        case 'Renault':
            return 'Alpine'
        case 'Racing Point':
            return 'Aston Martin'
        case _:
            return old_team_name

In [ ]:
def quali_info(season, file_name=None):

    schedule = ff1.get_event_schedule(season)
    num_rounds = schedule['RoundNumber'].max()
    
    quali_laps = []
    
    for rnd in range (1, num_rounds + 1):
        quali = ff1.get_session(season, rnd, 'Q')
        quali.load()
        laps = quali.laps
        results = quali.results
        drivers = [quali.get_driver(drv_num)['Abbreviation'] for drv_num in quali.drivers]
        
        for drv in drivers:
            drv_laps = laps.pick_drivers(drv)
            result = results[results['Abbreviation'] == drv].iloc[0]
            if len(drv_laps) > 0:
                fastest_lap = drv_laps.pick_fastest()
                if fastest_lap is not None and not pd.isnull(fastest_lap['LapTime']):
                    quali_laps.append({
                        'Season' : season,
                        'Round' : rnd,
                        'Season_Round' : f"{season}-{rnd:02d}",
                        'Grand_Prix' : quali.event['EventName'],
                        'Driver' : drv,
                        'Team' : current_team_name(fastest_lap['Team']),
                        'Quali_Position' : int(result['Position']),
                        'Quali_Time' : fastest_lap['LapTime'].total_seconds()})
                else:
                  print(f"Driver: {drv}, Grand Prix: {quali.event['EventName']} no valid quali laps")
            else:
                print(f"Driver: {drv}, Grand Prix: {quali.event['EventName']} no laps in quali")
    
    df_quali_info = pd.DataFrame(quali_laps)

    df_quali_info['Pole_Time'] = df_quali_info.groupby('Round')['Quali_Time'].transform('min')
    df_quali_info['Normalised_Pole_Gap'] = ((df_quali_info['Quali_Time'] / df_quali_info['Pole_Time']) - 1)
    df_quali_info['Pole_Gap_Position'] = (df_quali_info.groupby('Round')['Normalised_Pole_Gap'].rank(method = 'min', ascending = True).astype(int))

    df_quali_info = df_quali_info[[
        'Season',
        'Round',
        'Season_Round',
        'Grand_Prix',
        'Driver',
        'Team',
        'Quali_Position',
        'Pole_Gap_Position',
        'Pole_Time',
        'Quali_Time',
        'Normalised_Pole_Gap']]
    
    if file_name is None:
        return df_quali_info
    
    file = Path(f"{file_name}.csv")

    if file.exists():
        df_exist = pd.read_csv(file)
        df_add = pd.concat([df_exist, df_quali_info], ignore_index=True)
        df_add = df_add.drop_duplicates(subset=['Season_Round', 'Driver'], keep='last')
        df_add.to_csv(file, index=False)
    else:
        df_quali_info.to_csv(file, index=False)
    
    return df_quali_info

In [ ]:
quali_dataset = quali_info(season = 2022, file_name = 'X_season_quali_dataset')